In [4]:

from langchain_community.document_loaders import (
    DirectoryLoader,
    PyMuPDFLoader
)


def load_pdfs(directory: str):
    """Load PDF files from a directory and return Document objects."""

    loader = DirectoryLoader(
        path=directory,
        glob="**/*.pdf",
        loader_cls=PyMuPDFLoader,
        show_progress=True,
        silent_errors=False
    )

    documents = loader.load()

    if not documents:
        raise ValueError(f"No PDF documents found in: {directory}")

    print(f"Successfully loaded {len(documents)} PDF pages.")

    return documents


# Example usage
documents = load_pdfs("../data/ScienceDirect_articles_01Jan2026_18-06-18.925/")

print(documents[0].page_content)
print(documents[0].metadata)


print(documents[0].metadata)

100%|██████████| 57/57 [00:20<00:00,  2.74it/s]

Successfully loaded 554 PDF pages.
Computer Communications 154 (2020) 433–441
Contents lists available at ScienceDirect
Computer Communications
journal homepage: www.elsevier.com/locate/comcom
A decision based unsymmetrical trimmed modified winsorized variants for
the removal of high density salt and pepper noise in images and videos
Vasanth kishorebabu a,∗, R. Varatharajan b
a Department of E.C.E, Vidya Jyothi Institute of Technology, Aziz Nagar Gate, chilukur road, Hyderabad-78, Telengana, India
b Department of Electronics and Communication Engineering, School of Electronics Engineering Bharath University, Chennai, India
A R T I C L E
I N F O
Keywords:
Unsymmetrical trimmed modified winsorized
variants
Salt and pepper noise
Edge preservation
A B S T R A C T
A Decision Based Unsymmetrical trimmed modified Winsorized Variants for the removal of high density salt
and pepper noise in images and videos is proposed. The algorithm initially checks for fixed outliers (0 or
255) in the images

In [7]:

from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents, chunk_size=1000, chunk_overlap=200):
    """Split loaded documents into smaller chunks."""

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap
    )

    chunks = text_splitter.split_documents(documents)

    print(f"Created {len(chunks)} chunks.")

    return chunks


### Kutumia function


documents = load_pdfs("../data/ScienceDirect_articles_01Jan2026_18-06-18.925")

chunks = split_documents(
    documents,
    chunk_size=1000,
    chunk_overlap=200
)

print(chunks[1].metadata)
print(chunks[0].metadata)


100%|██████████| 57/57 [00:20<00:00,  2.76it/s]


Successfully loaded 554 PDF pages.
Created 3630 chunks.
{'producer': 'pdfTeX', 'creator': 'Elsevier', 'creationdate': '2020-03-19T18:35:02+05:30', 'source': '..\\data\\ScienceDirect_articles_01Jan2026_18-06-18.925\\A-decision-based-unsymmetrical-trimmed-modified-winsorized-va_2020_Computer-.pdf', 'file_path': '..\\data\\ScienceDirect_articles_01Jan2026_18-06-18.925\\A-decision-based-unsymmetrical-trimmed-modified-winsorized-va_2020_Computer-.pdf', 'total_pages': 9, 'format': 'PDF 1.7', 'title': '', 'author': '', 'subject': 'Computer Communications, 154 (2020) 433-441. doi:10.1016/j.comcom.2020.02.048', 'keywords': '', 'moddate': '2020-03-19T18:35:02+05:30', 'trapped': '', 'modDate': "D:20200319183502+05'30'", 'creationDate': "D:20200319183502+05'30'", 'page': 0}
{'producer': 'pdfTeX', 'creator': 'Elsevier', 'creationdate': '2020-03-19T18:35:02+05:30', 'source': '..\\data\\ScienceDirect_articles_01Jan2026_18-06-18.925\\A-decision-based-unsymmetrical-trimmed-modified-winsorized-va_2020_C

In [10]:

import os

from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_qdrant import QdrantVectorStore


class VectorStoreManager:

    def __init__(
        self,
        collection_name: str,
        qdrant_url: str,
        qdrant_api_key: str,
        embedding_model: str = "gemini-embedding-2",
    ):
        self.collection_name = collection_name

        # Gemini Embedding Model
        self.embeddings = GoogleGenerativeAIEmbeddings(
            model=embedding_model,
            google_api_key=os.getenv("GOOGLE_API_KEY"),
        )

        # Qdrant configuration
        self.qdrant_url = qdrant_url
        self.qdrant_api_key = qdrant_api_key

        self.vector_store = None

    def create_vector_store(self, documents):
        """Create Qdrant collection and store document embeddings."""

        self.vector_store = QdrantVectorStore.from_documents(
            documents=documents,
            embedding=self.embeddings,
            url=self.qdrant_url,
            api_key=self.qdrant_api_key,
            collection_name=self.collection_name,
        )

        print(
            f"Successfully stored {len(documents)} documents "
            f"in Qdrant collection '{self.collection_name}'."
        )

        return self.vector_store

    def get_retriever(self, k: int = 5):
        """Return Qdrant retriever."""

        if self.vector_store is None:
            raise ValueError(
                "Vector store has not been initialized. "
                "Call create_vector_store() first."
            )

        return self.vector_store.as_retriever(
            search_kwargs={"k": k}
        )
